In [1]:
from pathlib import Path
import sys
import pickle
import pandas as pd

# Add src/ so Python can find your modules
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

# Import your existing functions
from text_cleaning import basic_clean_text
from greenwashing_scoring import simple_greenwashing_score


In [2]:
# Load the saved logistic regression model
model_path = PROJECT_ROOT / "models" / "logreg_model.pkl"
with open(model_path, "rb") as f:
    model = pickle.load(f)

# Load the saved TF-IDF vectorizer
vectorizer_path = PROJECT_ROOT / "models" / "tfidf_vectorizer.pkl"
with open(vectorizer_path, "rb") as f:
    vectorizer = pickle.load(f)

"Model and vectorizer loaded!"


'Model and vectorizer loaded!'

In [3]:
def analyze_text(text: str):
    """
    Runs the full greenwashing analysis pipeline on a single text input.
    Returns keyword score, ML prediction, and cleaned text.
    """
    
    # 1. Clean text
    cleaned = basic_clean_text(text)
    
    # 2. Keyword-based score
    keyword_results = simple_greenwashing_score(text)
    
    # 3. ML-based prediction
    vec = vectorizer.transform([cleaned])
    ml_pred = model.predict(vec)[0]
    ml_prob = model.predict_proba(vec)[0][ml_pred]
    
    # Return everything together
    return {
        "original_text": text,
        "cleaned_text": cleaned,
        "keyword_score": keyword_results["score"],
        "keyword_risk_level": keyword_results["risk_level"],
        "matched_keywords": keyword_results["matched_keywords"],
        "ml_prediction": int(ml_pred),
        "ml_confidence": float(ml_prob)
    }


In [4]:
sample_text = """
Our products use eco-friendly, chemical-free, planet-loving ingredients 
that help create a sustainable future for everyone!
"""

results = analyze_text(sample_text)
results


{'original_text': '\nOur products use eco-friendly, chemical-free, planet-loving ingredients \nthat help create a sustainable future for everyone!\n',
 'cleaned_text': 'our products use eco friendly chemical free planet loving ingredients that help create a sustainable future for everyone',
 'keyword_score': 70,
 'keyword_risk_level': 'High',
 'matched_keywords': ['eco friendly', 'sustainable', 'chemical free'],
 'ml_prediction': 1,
 'ml_confidence': 0.5690374879107147}

In [5]:
def analyze_dataframe(df, text_column="text"):
    """
    Applies the full analysis pipeline to every row in a DataFrame.
    Returns a new DataFrame with all analysis fields.
    """
    results = df[text_column].apply(analyze_text)
    return pd.DataFrame(results.tolist())


In [6]:
df_test = pd.DataFrame({
    "text": [
        "Our eco-friendly bottles reduce environmental impact!",
        "Buy one get one free this weekend only!",
        "We are committed to sustainable operations and emission reduction.",
    ]
})

batch_results = analyze_dataframe(df_test)
batch_results


,original_text,cleaned_text,keyword_score,keyword_risk_level,matched_keywords,ml_prediction,ml_confidence
0,Our eco-friendly bottles reduce environmental ...,our eco friendly bottles reduce environmental ...,20,Low,[eco friendly],1,0.575217
1,Buy one get one free this weekend only!,buy one get one free this weekend only,0,Low,[],0,0.591496
2,We are committed to sustainable operations and...,we are committed to sustainable operations and...,20,Low,[sustainable],1,0.537824
